In [1]:
from pathlib import Path
from zipfile import ZipFile
from collections import defaultdict
import importlib.util
import subprocess
import sys
import warnings

needed = {"lxml": "lxml", "shapely": "shapely", "pyproj": "pyproj"}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from lxml import etree
from shapely.geometry import LineString, Point
from shapely.ops import transform, unary_union
from pyproj import Transformer
warnings.filterwarnings("ignore", message="The .shapely.ops.transform.* function is deprecated.*", category=DeprecationWarning)

lane_file = Path("movcud-zolaqlar-velo.kmz")
parking_file = Path("mpv-v-mps.-son.kmz")
if not lane_file.exists() or not parking_file.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("İki orijinal KMZ faylını notebook ilə eyni qovluğa qoyun.")
    files.upload()
if not lane_file.is_file() or not parking_file.is_file():
    raise FileNotFoundError("Orijinal KMZ faylları tapılmadı: " +
                            ", ".join(p.name for p in (lane_file, parking_file) if not p.is_file()))

ns = {"k": "http://www.opengis.net/kml/2.2"}

def layer_placemarks(path, layer_name):
    with ZipFile(path) as archive:
        kml = [name for name in archive.namelist() if name.lower().endswith(".kml")]
        if len(kml) != 1:
            raise ValueError(f"{path.name}: bir KML gözlənilirdi")
        root = etree.fromstring(archive.read(kml[0]))
    layers = [node for node in root.xpath(".//k:Folder | .//k:Document", namespaces=ns)
              if (node.findtext("k:name", namespaces=ns) or "").strip() == layer_name]
    if len(layers) != 1:
        raise ValueError(f"{layer_name}: bir qat gözlənilirdi, tapıldı {len(layers)}")
    return layers[0].xpath(".//k:Placemark", namespaces=ns)

def coordinates(node):
    return [tuple(map(float, token.split(",")[:2]))
            for token in (node.text or "").split()]

to_utm = Transformer.from_crs("EPSG:4326", "EPSG:32639", always_xy=True).transform
lane_marks = layer_placemarks(lane_file, "Mövcud zolaqlar")
park_marks = layer_placemarks(parking_file, "velosiped məntəqələri yeni")
lanes, parking, empty_lanes = [], [], []
for mark in lane_marks:
    name = (mark.findtext("k:name", namespaces=ns) or "").strip()
    parts = [coordinates(node) for node in mark.xpath(".//k:LineString/k:coordinates", namespaces=ns)]
    valid_parts = [part for part in parts if len(part) >= 2]
    if not valid_parts:
        # Mənbədə bir xəttin geometriyası boşdur; ölçülə bilən uzunluğa daxil edilmir.
        empty_lanes.append(name)
    for part in valid_parts:
        geom = transform(to_utm, LineString(part))
        if not geom.is_valid or geom.is_empty:
            raise ValueError(f"Etibarsız zolaq geometriyası: {name}")
        lanes.append((name, geom))
for mark in park_marks:
    name = (mark.findtext("k:name", namespaces=ns) or "").strip()
    points = [coordinates(node) for node in mark.xpath(".//k:Point/k:coordinates", namespaces=ns)]
    if len(points) != 1 or len(points[0]) != 1 or not name:
        raise ValueError(f"Etibarsız parkinq nöqtəsi: {name}")
    lon, lat = points[0][0]
    parking.append((name, lon, lat, Point(*to_utm(lon, lat))))

assert len(lanes) == 60 and len(empty_lanes) == 1
assert len(parking) == 185 and len({name for name, *_ in parking}) == len(parking)
assert len({(lon, lat) for _, lon, lat, _ in parking}) == len(parking)
assert len({geom.wkb for _, geom in lanes}) == len(lanes)
lane_network = unary_union([geom for _, geom in lanes])

## Sual 1.1

In [2]:
lengths = defaultdict(float)
for street, geometry in lanes:
    lengths[street] += geometry.length
total_km = sum(geometry.length for _, geometry in lanes) / 1000
top_street = max(lengths, key=lengths.get)
print(f"{total_km:.2f}, {top_street}")

57.14, 8 Noyabr pr.


## Sual 1.2

In [3]:
distances = {name: point.distance(lane_network) for name, _, _, point in parking}
covered = sum(distance <= 50 for distance in distances.values())
assert covered == sum(any(point.distance(line) <= 50 for _, line in lanes)
                      for _, _, _, point in parking)
print(covered)

87


## Sual 1.3

In [4]:
outside = [name for name, lon, lat, _ in parking
           if not (40.30 <= lat <= 40.46 and 49.70 <= lon <= 50.05)]
print(f"{len(outside)}: {', '.join(outside)}")

4: MPV-144-2, MPV-145-2, MPV-146-2, MPV-147-2


## Sual 4

In [5]:
# Mövcud Vaqif prospekti zolağı AYNA qatında təsdiqlənir.
# AYNA: https://ayna.gov.az/az/news/sefayet-mehdiyev-kucesi-vaqif-ve-metbuat-prospektlerinde-mikromobillik-zolaqlari-teskil-edilib-1105
# Küçə üzrə yeni xətt geometriyası olmadığı üçün tikinti uzunluğu və yeni 50 m əhatəsi iddia edilmir.
assert "Vaqif pr." in lengths
assert any(distance > 50 for name, lon, lat, _ in parking
           if 49.835 <= lon <= 49.855 and 40.393 <= lat <= 40.415
           for distance in [distances[name]])
recommendation = ("Vaqif prospektindəki mövcud velosiped zolağının Atatürk parkı "
                  "istiqamətində davam etdirilməsini təklif edirəm. AYNA-nın parkinq "
                  "məlumatlarında bu ərazidə mövcud zolaqlardan uzaq məntəqələr görünür. "
                  "Prospektdə artıq zolaq olduğuna görə yeni hissə həmin xəttə bağlanaraq "
                  "parkinqlərin şəbəkəyə qoşulması üçün əsas yarada bilər. AYNA da parkla "
                  "əlaqəni gələcək mərhələ kimi qeyd edib. Dəqiq marşrut və kəsişmələrin "
                  "uyğunluğu layihələndirmə zamanı yoxlanmalıdır.")
assert len(recommendation.split()) <= 99
print(recommendation)

Vaqif prospektindəki mövcud velosiped zolağının Atatürk parkı istiqamətində davam etdirilməsini təklif edirəm. AYNA-nın parkinq məlumatlarında bu ərazidə mövcud zolaqlardan uzaq məntəqələr görünür. Prospektdə artıq zolaq olduğuna görə yeni hissə həmin xəttə bağlanaraq parkinqlərin şəbəkəyə qoşulması üçün əsas yarada bilər. AYNA da parkla əlaqəni gələcək mərhələ kimi qeyd edib. Dəqiq marşrut və kəsişmələrin uyğunluğu layihələndirmə zamanı yoxlanmalıdır.
